<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/04_Expert/L48_State_Space_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L48: State Space Models - S4, Mamba, and Beyond Attention

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Expert  
**Lesson:** 48 of 60

---

## Learning Objectives

By the end of this lesson, you will:
- Understand state space models (SSM): continuous-time recurrence, discretization
- Implement a minimal linear SSM layer (recurrent form)
- Compare SSMs to attention: O(L) vs O(L^2), long-range dependency

---

## Concept: State Space Models

**SSMs** model sequences with a hidden state: h' = A h + B x, y = C h. **S4** and **Mamba** are efficient parameterizations; Mamba adds input-dependent gates. They offer O(L) complexity and long context. We implement a simple discrete SSM step.

---

In [ ]:
import torch
import torch.nn as nn
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: Discrete SSM Step

---

In [ ]:
class SimpleSSMLayer(nn.Module):
    def __init__(self, d_model, d_state=64):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.A = nn.Parameter(torch.randn(d_state, d_state) * 0.01)
        self.B = nn.Linear(d_model, d_state)
        self.C = nn.Linear(d_state, d_model)
        self.D = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, C_in = x.shape
        h = torch.zeros(B, self.d_state, device=x.device)
        outs = []
        for t in range(T):
            h = h + self.B(x[:, t]) * 0.1
            h = h @ self.A
            y_t = self.C(h) + self.D(x[:, t])
            outs.append(y_t)
        return torch.stack(outs, dim=1)

ssm = SimpleSSMLayer(64, 32)
x = torch.randn(2, 10, 64)
y = ssm(x)
print("SSM output shape:", y.shape)
print("Complexity: O(L) per layer vs O(L^2) for attention.")

## Part 2: S4 / Mamba (Concept)

---

In [ ]:
print("S4: HiPPO initialization for A, fast convolution via FFT.")
print("Mamba: input-dependent A,B,C; selective scan for efficiency.")
print("Libraries: mamba-ssm, causal-conv1d for production Mamba.")

## Exercises

1. Run the SSM on sequence length 100 vs 1000; compare time to a single attention layer.
2. Install mamba-ssm and run a small Mamba model if GPU allows.
3. Read the Mamba paper and explain "selective" state space.

---

## Key Takeaways

1. SSMs give O(L) sequence modeling; good for long context.
2. S4/Mamba are efficient SSM variants used in language and vision.
3. Can replace or complement attention in hybrid architectures.

---

## Next Lesson

**L49: Advanced Retrieval** – Dense retrieval, re-ranking, hybrid search.

---